### Unit Test 03: Deduplication Logic Validation

This unit test validates the deduplication logic used in the pipeline to prevent duplicate records during ingestion and transformation.

The test creates a small sample dataset containing duplicate records for the same `transaction_id` with different `silver_updated_at` timestamps. It then applies the deduplication logic using a window function and verifies that:

- Only one record is retained per business key (`transaction_id`)
- The latest record is selected based on the most recent `silver_updated_at`
- Output data matches the expected DataFrame using `assertDataFrameEqual`

This ensures the pipeline deduplication approach is correct and reusable across tables where incremental updates are expected.


In [0]:

# Validates deduplication logic by ensuring only the latest record is retained
# for the same business key based on the updated timestamp.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.testing.utils import assertDataFrameEqual

# Sample input with duplicates for the same transaction_id
df = spark.createDataFrame(
    [
        (101, "U01", "2026-02-01 10:00:00", "2026-02-01 10:01:00"),
        (101, "U01", "2026-02-01 10:00:00", "2026-02-01 10:05:00"),  # latest
        (102, "U02", "2026-02-01 11:00:00", "2026-02-01 11:02:00"),
    ],
    ["transaction_id", "user_id", "created_at", "silver_updated_at"]
).withColumn("created_at", F.to_timestamp("created_at")) \
 .withColumn("silver_updated_at", F.to_timestamp("silver_updated_at"))

# Dedup logic: keep latest row per transaction_id based on silver_updated_at
w = Window.partitionBy("transaction_id").orderBy(F.col("silver_updated_at").desc())

df_dedup = (
    df.withColumn("_rn", F.row_number().over(w))
      .filter(F.col("_rn") == 1)
      .drop("_rn")
)

# Expected output: only latest record for transaction_id = 101 should remain
expected_df = spark.createDataFrame(
    [
        (101, "U01", "2026-02-01 10:00:00", "2026-02-01 10:05:00"),
        (102, "U02", "2026-02-01 11:00:00", "2026-02-01 11:02:00"),
    ],
    ["transaction_id", "user_id", "created_at", "silver_updated_at"]
).withColumn("created_at", F.to_timestamp("created_at")) \
 .withColumn("silver_updated_at", F.to_timestamp("silver_updated_at"))

# Sort both before comparison for stable test results
df_dedup_sorted = df_dedup.orderBy("transaction_id")
expected_sorted = expected_df.orderBy("transaction_id")

assertDataFrameEqual(df_dedup_sorted, expected_sorted)

print("Unit Test 03 passed")
